# M5 · Derivatives and the Chain Rule — companion notebook

> **Play with this.** A *demonstration, not an assessment* — the module's real assessment is its problem set. This notebook checks derivatives numerically (nudge and measure), watches finite differences fail when the step is wrong — a good scare — verifies the chain rule link by link, walks a gradient uphill, and runs the module's computational graph as thirty lines of hand-rolled reverse-mode autodiff.

Companion to the **Derivatives and the Chain Rule** module of the Mathematical Foundations track at [llmsforsocialscience.net](https://llmsforsocialscience.net/).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5)

## 1 · A derivative is a nudge, measured

The definition is executable: nudge the input, divide the output movement by the nudge. Compare against the analytic answer.

In [ ]:
f = lambda x: np.exp(-x**2)          # the function
df = lambda x: -2*x*np.exp(-x**2)    # its analytic sensitivity (problem 1b)

x0, h = 0.7, 1e-6
numeric = (f(x0 + h) - f(x0)) / h
print(f"numeric  (nudge & measure): {numeric:.8f}")
print(f"analytic (chain rule):      {df(x0):.8f}")

## 2 · The scare: finite differences fail if you nudge wrongly

Sweep the nudge size from large to absurdly small. Too large: the linear approximation breaks. Too small: floating-point cancellation destroys the numerator. There is a valley of usable step sizes — and this fragility is *why autodiff exists*: the chain rule computes exact sensitivities with no step size at all.

In [ ]:
hs = np.logspace(0, -14, 60)
errors = [abs((f(x0 + h) - f(x0)) / h - df(x0)) for h in hs]

plt.figure(figsize=(8, 4))
plt.loglog(hs, errors)
plt.gca().invert_xaxis()
plt.xlabel("nudge size h  (shrinking →)")
plt.ylabel("error vs analytic derivative")
plt.title("Finite differences: too big fails, too small also fails")
plt.tight_layout(); plt.show()

for h in [1e-1, 1e-6, 1e-13]:
    print(f"h = {h:6.0e}:  error = {abs((f(x0+h)-f(x0))/h - df(x0)):.2e}")

## 3 · The chain rule, verified link by link

$y = \log(1 + e^{3x})$ is a three-link chain: $x \to 3x \to 1 + e^{u} \to \log v$. Measure each link's local rate numerically, multiply them, and compare with the end-to-end rate — rates through a chain multiply.

In [ ]:
x0, h = 0.4, 1e-7
g1 = lambda x: 3*x
g2 = lambda u: 1 + np.exp(u)
g3 = lambda v: np.log(v)

u0, v0 = g1(x0), g2(g1(x0))
local1 = (g1(x0+h) - g1(x0)) / h
local2 = (g2(u0+h) - g2(u0)) / h
local3 = (g3(v0+h) - g3(v0)) / h
end_to_end = (g3(g2(g1(x0+h))) - g3(g2(g1(x0)))) / h

print(f"local rates:      {local1:.5f} x {local2:.5f} x {local3:.5f}")
print(f"their product:    {local1*local2*local3:.5f}")
print(f"end-to-end rate:  {end_to_end:.5f}")

## 4 · Long chains run to extremes

Problem 5, run live: compose many links whose local rates hover below or above one, and watch the end-to-end sensitivity vanish or explode. The failure modes of D7's recurrent networks, in four lines.

In [ ]:
for rate, name in [(0.5, "vanishing"), (0.9, "slowly vanishing"), (1.0, "stable"), (1.1, "slowly exploding"), (2.0, "exploding")]:
    products = rate ** np.arange(1, 31)
    plt.semilogy(products, label=f"local rate {rate} ({name})")
plt.xlabel("chain length"); plt.ylabel("end-to-end sensitivity (log scale)")
plt.legend(fontsize=8); plt.title("Products of many local rates run to extremes")
plt.tight_layout(); plt.show()

## 5 · The gradient points uphill — walk it

A two-variable function, its gradient field, and a short walk taking steps along the gradient. The walker climbs — and the contour plot shows why: the gradient is always perpendicular to the level curves, the direction in which the function has no choice but to change fastest.

In [ ]:
F  = lambda x, y: -(x**2 + 2*y**2)          # a smooth hill, peak at the origin
Fx = lambda x, y: -2*x
Fy = lambda x, y: -4*y

xs = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(xs, xs)

path = [np.array([-1.8, 1.5])]
for _ in range(25):
    p = path[-1]
    grad = np.array([Fx(*p), Fy(*p)])
    path.append(p + 0.08 * grad)             # step UPhill, along the gradient
path = np.array(path)

plt.figure(figsize=(6.5, 5.5))
plt.contour(X, Y, F(X, Y), levels=14, linewidths=0.6)
s = np.linspace(-2, 2, 15)
SX, SY = np.meshgrid(s, s)
plt.quiver(SX, SY, Fx(SX, SY), Fy(SX, SY), alpha=0.35, width=0.003)
plt.plot(path[:, 0], path[:, 1], "o-", color="tab:red", ms=3, label="gradient ascent")
plt.legend(); plt.title("Steps along the gradient climb the hill")
plt.gca().set_aspect("equal"); plt.tight_layout(); plt.show()

print(f"start height: {F(*path[0]):.3f}   end height: {F(*path[-1]):.3f}   (0 is the peak)")

## 6 · The widget's graph, as thirty lines of autodiff

The module's computational graph, $L = (wx + b - y)^2$, executed exactly as the widget steps it: forward values, then backward, every step *arrived × local*. This is a complete (tiny) reverse-mode autodiff engine — the same numbers as the widget, and the same procedure a deep-learning framework runs on a billion parameters.

In [ ]:
def graph(w, x, b, y):
    # forward
    m = w * x
    s = m + b
    r = s - y
    L = r ** 2
    # backward: every line is (arrived from downstream) x (local derivative)
    dL = 1.0
    dr = dL * 2 * r          # square node: local rate 2r
    ds = dr * 1              # subtraction w.r.t. s: local rate 1
    dy = dr * -1             # subtraction w.r.t. y: local rate -1
    dm = ds * 1              # add node copies
    db = ds * 1
    dw = dm * x              # multiply node: local rate = the other input
    dx = dm * w
    return L, dict(w=dw, x=dx, b=db, y=dy)

L, grads = graph(w=2.0, x=3.0, b=1.0, y=5.0)
print(f"L = {L}")
for k, v in grads.items():
    print(f"dL/d{k} = {v}")

# check every gradient against nudge-and-measure
h = 1e-6
for k in ["w", "x", "b", "y"]:
    args = dict(w=2.0, x=3.0, b=1.0, y=5.0)
    args[k] += h
    numeric = (graph(**args)[0] - L) / h
    print(f"dL/d{k}: backward {grads[k]:8.3f}   numeric {numeric:8.3f}")

Every backward number matches its nudge-and-measure check — with no step size to tune, because the chain rule is exact. A framework's `.backward()` is this cell with better bookkeeping. *(The module spec suggests re-running this graph in an autodiff library; this staging draft stays numpy-only by house constraint — in Colab, `torch` reproduces the numbers in five lines if you want the cross-check.)*

---

**Next:** M6 · Optimisation — the gradient gives every parameter a direction; M6 is about the one knob that decides how far to step, and everything that goes wrong with it.